# GA Random Search on Kaggle

Attach two Kaggle inputs before running:

- A source-code dataset containing `random_search_ga.py`, `GA.py`, `eval.py`, `load_data.py`, and the model files.
- A data dataset containing `RBA.xlsx`.

The notebook creates a smaller balanced dataset in `/kaggle/working` and runs the existing `random_search_ga.py` with `--test AUTOML`.

In [ ]:
# 1. Install only missing lightweight dependencies
import importlib
import subprocess
import sys

def ensure(import_name, package_name=None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name or import_name])

ensure('deap', 'deap')
ensure('openpyxl', 'openpyxl')

print('Dependencies ready.')


In [ ]:
# 2. Copy project source files to /kaggle/working
from pathlib import Path
import shutil
import sys

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

required = {'random_search_ga.py', 'GA.py', 'eval.py', 'load_data.py', 'MLP.py', 'CNN.py', 'DNN.py', 'RNN.py', 'LSTM.py'}
source_dir = None
for candidate in INPUT_DIR.rglob('random_search_ga.py'):
    if required.issubset({p.name for p in candidate.parent.glob('*.py')}):
        source_dir = candidate.parent
        break

if source_dir is None:
    raise FileNotFoundError(
        'Could not find the source-code dataset. Attach a Kaggle dataset containing random_search_ga.py, GA.py, eval.py, load_data.py, and model files.'
    )

for src in source_dir.glob('*.py'):
    shutil.copy2(src, WORK_DIR / src.name)

sys.path.insert(0, str(WORK_DIR))
print(f'Copied source files from: {source_dir}')
print(f'Working directory: {WORK_DIR}')


In [ ]:
# 3. Create a smaller balanced RBA dataset for faster random-search runs
from pathlib import Path
import pandas as pd

ROWS_PER_CLASS = 500  # increase to 1000 or more after the first test run
SEED = 43

rba_candidates = list(Path('/kaggle/input').rglob('RBA.xlsx'))
if not rba_candidates:
    raise FileNotFoundError('Could not find RBA.xlsx in /kaggle/input. Attach the ransomware/RBA dataset first.')

RBA_SOURCE = rba_candidates[0]
SMALL_DATA_PATH = Path('/kaggle/working/RBA_small_random_search.xlsx')

df = pd.read_excel(RBA_SOURCE, engine='openpyxl')
if 'Class' not in df.columns:
    raise KeyError("RBA.xlsx must contain the target column 'Class'.")

parts = []
for label, group in df.groupby('Class', sort=False):
    n = min(ROWS_PER_CLASS, len(group))
    parts.append(group.sample(n=n, random_state=SEED))

small = pd.concat(parts, ignore_index=False).sample(frac=1, random_state=SEED).reset_index(drop=True)
small.to_excel(SMALL_DATA_PATH, index=False, engine='openpyxl')

print(f'Source: {RBA_SOURCE}')
print(f'Smaller dataset: {SMALL_DATA_PATH}')
print(f'Shape: {small.shape}')
print(small['Class'].value_counts().to_string())


In [ ]:
# 4. Random-search settings
# Keep this small for the first Kaggle run. Increase after confirming everything works.
N_CONFIGS = 1
RUNS_PER_CONFIG = 1
FIXED_EPOCHS = 30
CV_FOLDS = 5
BASE_SEED = 43
RUN_FULL_SEARCH = False  # set True in the next cell when ready

OUTPUT_DIR = '/kaggle/working/random_search_ga_results'

base_cmd = [
    sys.executable,
    '/kaggle/working/random_search_ga.py',
    '--data-path', str(SMALL_DATA_PATH),
    '--dataset-idx', '1',
    '--test', 'AUTOML',
    '--n-configs', str(N_CONFIGS),
    '--runs-per-config', str(RUNS_PER_CONFIG),
    '--fixed-epochs', str(FIXED_EPOCHS),
    '--cv-folds', str(CV_FOLDS),
    '--base-seed', str(BASE_SEED),
    '--output-dir', OUTPUT_DIR,
]

print(' '.join(base_cmd))


In [ ]:
# 5. Dry run: confirms the script sees the smaller dataset path and AUTOML mode
import subprocess

subprocess.run(base_cmd + ['--dry-run'], check=True)


In [ ]:
# 6. Full run
if RUN_FULL_SEARCH:
    subprocess.run(base_cmd, check=True)
else:
    print('Set RUN_FULL_SEARCH = True in cell 4, then run this cell to start training.')


In [ ]:
# 7. Inspect and package outputs
from pathlib import Path
import shutil

out = Path(OUTPUT_DIR)
if out.exists():
    for path in sorted(out.glob('*')):
        print(path)
    shutil.make_archive('/kaggle/working/random_search_ga_results', 'zip', out)
    print('Zipped results: /kaggle/working/random_search_ga_results.zip')
else:
    print('No output directory yet. Run the full search first.')
